# Analise e tratamento dados bronze b_contas_receber.csv

In [1]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
BRONZE_DIR =  os.path.join(DATA_DIR, 'bronze')

In [3]:
df = pd.read_csv(os.path.join(BRONZE_DIR, 'b_contas_receber.csv'))
df.shape

(507, 9)

## Analise exploratória

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_receber  507 non-null    str    
 1   id_cliente         507 non-null    str    
 2   data_emissao       507 non-null    str    
 3   data_vencimento    507 non-null    str    
 4   data_recebimento   507 non-null    str    
 5   valor_titulo       507 non-null    float64
 6   valor_recebido     507 non-null    float64
 7   status             507 non-null    str    
 8   forma_recebimento  507 non-null    str    
dtypes: float64(2), str(7)
memory usage: 35.8 KB


In [6]:
df.isnull().sum()

id_titulo_receber    0
id_cliente           0
data_emissao         0
data_vencimento      0
data_recebimento     0
valor_titulo         0
valor_recebido       0
status               0
forma_recebimento    0
dtype: int64

## Tratamento de dados

In [7]:
df_original = df.copy()

In [8]:
# campos datas

df['data_emissao'] = pd.to_datetime(df['data_emissao'], format='mixed')
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], format='mixed')
df['data_recebimento'] = pd.to_datetime(df['data_recebimento'], format='mixed')


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_titulo_receber  507 non-null    str           
 1   id_cliente         507 non-null    str           
 2   data_emissao       507 non-null    datetime64[us]
 3   data_vencimento    507 non-null    datetime64[us]
 4   data_recebimento   507 non-null    datetime64[us]
 5   valor_titulo       507 non-null    float64       
 6   valor_recebido     507 non-null    float64       
 7   status             507 non-null    str           
 8   forma_recebimento  507 non-null    str           
dtypes: datetime64[us](3), float64(2), str(4)
memory usage: 35.8 KB


### Criando campos extras

In [10]:
# qtde_dias_emissao

df['qtde_dias_emissao'] = df['data_vencimento'] - df['data_emissao']


In [11]:
# qtde_dias_recebimento_atraso

df['qtde_dias_recebimento_atraso'] = df['data_recebimento'] - df['data_vencimento']


In [12]:
df.head()

,id_titulo_receber,id_cliente,data_emissao,data_vencimento,data_recebimento,valor_titulo,valor_recebido,status,forma_recebimento,qtde_dias_emissao,qtde_dias_recebimento_atraso
0,CR000001,C0027,2026-02-26,2026-04-12,2026-04-12,10168.68,10168.68,Pago,PIX,45 days,0 days
1,CR000002,C0012,2026-04-23,2026-06-22,2026-06-26,9370.77,9370.77,Pago,Boleto,60 days,4 days
2,CR000003,C0028,2026-04-14,2026-06-13,1900-01-01,914.85,0.00,Em aberto,Transferência,60 days,-46184 days
3,CR000004,C0026,2026-06-06,2026-07-21,1900-01-01,8707.44,0.00,Em aberto,PIX,45 days,-46222 days
4,CR000005,C0034,2026-01-28,2026-02-25,2026-02-25,7016.44,7016.44,Pago,Cartão,28 days,0 days


## Salvando dados Contas Receber em Silver

In [13]:
SILVER_DIR = os.path.join(DATA_DIR, 'silver')

In [14]:
df.to_excel(
    os.path.join(SILVER_DIR, 's_contas_receber.xlsx')
    , index=False
    , sheet_name='base_contas_receber'
)